# The Python Machine Learning Stack
## Machine Learning — BS Computer Science 3(3-0)



| | |
|---|---|
| **Notebook** | `NB02_Python_ML_Stack.ipynb` |
| **Lecture** | Lecture 3 (Week 1) |
| **CLO** | CLO-1 / CLO-5 |
| **Dataset** | Iris, plus a real CSV of your choice |

**Objective.** Install the working habits — vectorised NumPy, pandas inspection, labelled plots, the scikit-learn API — and see overfitting for the first time.

**By the end of this notebook you will be able to:**

1. Use vectorised NumPy operations and explain the axis argument.
2. Perform the six-step first look at any dataset with pandas.
3. Produce fully labelled histograms and class-coloured scatter plots.
4. Apply the fit / predict / score API inside a Pipeline and run a controlled experiment.

---

## 0. Setup and reproducibility

In [ ]:
# ---------------------------------------------------------------------
# Reproducibility header — required in EVERY notebook you submit
# ---------------------------------------------------------------------
import sys, numpy as np, pandas as pd, sklearn, matplotlib
import matplotlib.pyplot as plt

print('python      ', sys.version.split()[0])
print('numpy       ', np.__version__)
print('pandas      ', pd.__version__)
print('scikit-learn', sklearn.__version__)
print('matplotlib  ', matplotlib.__version__)

RANDOM_STATE = 1
rng = np.random.default_rng(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. NumPy — vectorise, never loop

A NumPy array stores one type in one contiguous block, so arithmetic runs in compiled C. **`axis=0` collapses rows and gives one value per column** — that is almost always what you want. Mixing up the axis is the most common NumPy bug in student code.

In [ ]:
a = np.array([[1., 2., 3.],
              [4., 5., 6.]])          # shape (2, 3)

print('shape       ', a.shape, ' dtype', a.dtype)
print('mean axis=0 ', a.mean(axis=0), '  <- per COLUMN (one per feature)')
print('mean axis=1 ', a.mean(axis=1), '        <- per ROW')
print('a * 2       ', a[0] * 2)
print('column 1    ', a[:, 1])
print('mask a > 3  ', a[a > 3])

z = (a - a.mean(axis=0)) / a.std(axis=0)      # standardisation
print('standardised:\n', np.round(z, 3))

### Exercise 1.1 — loop versus vectorised

Compute the column means of a 100,000 × 20 array twice: once with a Python loop and once with `.mean(axis=0)`. Time both with `%%timeit` and report the ratio.

In [ ]:
big = rng.normal(size=(100_000, 20))

import time

# --- Python loop version ---
start = time.perf_counter()
n_rows, n_cols = big.shape
loop_means = np.zeros(n_cols)
for j in range(n_cols):
    total = 0.0
    for i in range(n_rows):
        total += big[i, j]
    loop_means[j] = total / n_rows
loop_time = time.perf_counter() - start
print('loop time      : %.4f s' % loop_time)

# --- Vectorised version ---
start = time.perf_counter()
vec_means = big.mean(axis=0)
vec_time = time.perf_counter() - start
print('vectorised time: %.6f s' % vec_time)

print('results match  :', np.allclose(loop_means, vec_means))
print('speed ratio    : %.0fx faster' % (loop_time / vec_time))

> The vectorised `.mean(axis=0)` call runs through compiled C loops under the hood, making it hundreds to thousands of times faster than the equivalent Python-level loop (the exact ratio depends on the machine). This is why numerical computation should always use vectorised NumPy operations instead of raw Python loops.

## 2. pandas — the six-step first look

Run these on a **real, messy CSV**, not on Iris. What you are hunting for: shape, missing values, impossible values, class balance and feature scales.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

print(df.shape)                                   # 1. big enough?
display(df.head())                                # 2. always look
df.info()                                         # 3. dtypes, non-null
display(df.describe().T[['min', 'max', 'mean', 'std']].round(2))   # 4. impossible values?
print(df['target'].value_counts(normalize=True))  # 5. class balance
print(df.isnull().sum().sum(), 'missing values')  # 6. missingness

**Interpretation.** Look at the `min`/`max` column of `describe()`. Which two features have the most different scales, and by what factor? Which models from Week 3 onwards would that break?

> `proline` (range 278 to 1680) and `nonflavanoid_phenols` (range 0.13 to 0.66) have the most different scales — the maximum value of `proline` is roughly 2,500 times larger than the maximum value of `nonflavanoid_phenols`. Distance-based and gradient-based models such as **k-NN, SVM, logistic/linear regression (with regularization), and neural networks** would break on this, because large-range features (proline, magnesium) would dominate the distance or gradient calculation while small-range features are effectively ignored — this is exactly why scaling (e.g. `StandardScaler`) is necessary.

## 3. Matplotlib — the four plots you will draw all semester

Histogram, class-coloured scatter, line-versus-hyperparameter, bar chart. **Every plot needs axis labels with units and a legend.** An unlabelled plot carries no information and earns no marks.

In [ ]:
X, y = wine.data, wine.target

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

ax[0].hist(X[:, 0], bins=25, edgecolor='black')
ax[0].set_xlabel('alcohol [% vol]')
ax[0].set_ylabel('count')
ax[0].set_title('Histogram — distribution of one feature')

for cl, m in zip(np.unique(y), ('o', 's', '^')):
    ax[1].scatter(X[y == cl, 0], X[y == cl, 9], marker=m,
                  alpha=0.8, edgecolor='k', label=f'class {cl}')
ax[1].set_xlabel('alcohol [% vol]')
ax[1].set_ylabel('colour intensity')
ax[1].set_title('Scatter — are the classes separable?')
ax[1].legend()

plt.tight_layout(); plt.show()

## 4. Your first complete machine-learning program

Eleven lines. Every one of them corresponds to a concept from Lecture 2.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

pipe = Pipeline([('scaler', StandardScaler()),
                 ('knn', KNeighborsClassifier(n_neighbors=5))])
pipe.fit(X_train, y_train)

print('train accuracy %.3f' % pipe.score(X_train, y_train))
print('test  accuracy %.3f' % pipe.score(X_test,  y_test))

> **The most important line on this page.** The scaler is fitted on the *training data only* — that is what putting it inside the `Pipeline` guarantees. Fitting it on all the data lets the test set influence the training-time mean and standard deviation. That is **data leakage** and Week 8 is devoted to it.

## 5. Controlled experiment — change one thing at a time

Record the results in the table **before** you interpret anything. This is the standard experimental protocol for the rest of the course.

In [ ]:
rows = []
for k in (1, 5, 25, 100):
    p = Pipeline([('scaler', StandardScaler()),
                  ('knn', KNeighborsClassifier(n_neighbors=k))])
    p.fit(X_train, y_train)
    rows.append({'variant': 'scaled',   'k': k,
                 'train': p.score(X_train, y_train),
                 'test':  p.score(X_test,  y_test)})

    raw = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    rows.append({'variant': 'unscaled', 'k': k,
                 'train': raw.score(X_train, y_train),
                 'test':  raw.score(X_test,  y_test)})

results = pd.DataFrame(rows).pivot(index='k', columns='variant',
                                   values=['train', 'test']).round(3)
display(results)

### Interpretation — three things to explain

1. At `k = 1` the **training** accuracy is exactly 1.000. Why is that guaranteed, and what does the number tell you about the model?
2. What happened when you removed the scaler, and why does k-NN care about units?
3. What happens to the test accuracy as `k` approaches the size of the training set?

> **1.** At `k=1`, each training point's nearest neighbour is itself (distance = 0), so the model always predicts its own label back — training accuracy of 1.000 is therefore guaranteed. This tells us the model has simply **memorised** the training data rather than learning a generalisable pattern; it is a classic sign of overfitting.
>
> **2.** Removing the scaler drops accuracy, because k-NN is based on Euclidean distance and large-range features (e.g. proline, magnesium) dominate that distance calculation — small-range features (e.g. hue, nonflavanoid_phenols) end up contributing almost nothing. k-NN therefore needs every feature on a comparable scale to work correctly.
>
> **3.** As `k` approaches the size of the training set, the model averages over more and more neighbours and starts predicting close to the **majority class** regardless of the input — both train and test accuracy fall toward the majority-class baseline, a sign of underfitting.

---
## Deliverable

The timing exercise (1.1), the six-step first look on a real CSV, two fully labelled plots, and the four-row results table with all three interpretations written out.

---
## Submission checklist

Tick every box before you submit. Items 1–6 are the course reproducibility
rules from Lecture 3 and are marked in the rubric.

- [ ] The random seed is fixed **and printed**
- [ ] Library versions are printed in the first cell
- [ ] The raw data file was **not** modified
- [ ] Preprocessing happens **after** the split, inside a `Pipeline`
- [ ] The dataset source and licence are stated in a markdown cell
- [ ] **Kernel → Restart & Run All** completes without error
- [ ] Every experiment is followed by a written interpretation
- [ ] At least one wrong prediction or failure case is explained
- [ ] Results are reported as **mean ± standard deviation** where cross-validated
- [ ] Every plot has axis labels with units and a legend
